In [1]:
import sys; sys.path.append("..")
import numpy as np, pandas as pd, plotly.express as px
from src.data.loading import load_processed
from src.features.build import make_features, FEATURES

reg = pd.read_csv("../reports/house_registry.csv")
model_ids = reg.loc[reg.group == "model", "house_id"].tolist()

data = pd.concat([make_features(load_processed(h, "hourly", "../data/processed"), h)
                  for h in model_ids])
data.index.name = "timestamp"
print(len(model_ids), "houses |", data.shape)

34 houses | (296761, 13)


In [2]:
SPLIT = pd.Timestamp("2019-03-01")

# Keep only rows with a real target and the main lags available (fair for every model)
valid = data.dropna(subset=["kwh", "lag_24", "lag_168"])
train = valid[valid.index < SPLIT]
test = valid[valid.index >= SPLIT].copy()

print(f"train {len(train):,} rows: {train.index.min().date()} to {train.index.max().date()}")
print(f"test  {len(test):,} rows: {test.index.min().date()} to {test.index.max().date()}")

train 214,375 rows: 2018-06-08 to 2019-02-28
test  74,335 rows: 2019-03-01 to 2019-05-31


In [3]:
def score(y, pred):
    err = y - pred
    return pd.Series({"MAE": err.abs().mean(),
                      "RMSE": np.sqrt((err ** 2).mean()),
                      "WAPE_%": 100 * err.abs().sum() / y.sum()})

BASELINES = {"Naive: yesterday": "lag_24",
             "Naive: last week": "lag_168",
             "Naive: 7-day same-hour average": "same_hour_7d_mean"}

results = {name: score(test.kwh, test[col]) for name, col in BASELINES.items()}
pd.DataFrame(results).T.round(3)

,MAE,RMSE,WAPE_%
Naive: yesterday,0.280,0.534,37.118999
Naive: last week,0.329,0.596,43.655998
Naive: 7-day same-hour average,0.251,0.453,33.321999


In [4]:
import lightgbm as lgb

model = lgb.LGBMRegressor(
    n_estimators=400, learning_rate=0.05, num_leaves=31,
    min_child_samples=50, subsample=0.8, subsample_freq=1,
    colsample_bytree=0.8, random_state=42, verbose=-1,
)
model.fit(train[FEATURES], train["kwh"])

test["pred"] = model.predict(test[FEATURES]).clip(min=0)   # energy can't be negative
results["LightGBM"] = score(test.kwh, test.pred)

In [5]:
res = pd.DataFrame(results).T.round(3)
best_naive = res.filter(like="Naive", axis=0)["MAE"].min()
res["MAE_gain_vs_best_naive_%"] = (100 * (1 - res["MAE"] / best_naive)).round(1)
res

,MAE,RMSE,WAPE_%,MAE_gain_vs_best_naive_%
Naive: yesterday,0.280,0.534,37.119,-11.6
Naive: last week,0.329,0.596,43.656,-31.1
Naive: 7-day same-hour average,0.251,0.453,33.322,0.0
LightGBM,0.243,0.436,32.287,3.2


In [6]:
best_col = BASELINES[res.filter(like="Naive", axis=0)["MAE"].idxmin()]

test["err_naive"] = (test.kwh - test[best_col]).abs()
test["err_model"] = (test.kwh - test.pred).abs()
ph = test.groupby("house_id")[["err_naive", "err_model"]].mean()
ph["gain_%"] = 100 * (1 - ph.err_model / ph.err_naive)

print(f"Model beats the best baseline in {(ph['gain_%'] > 0).sum()} of {len(ph)} houses")
ph.sort_values("gain_%").round(3).head(8)    # the houses where it does worst

Model beats the best baseline in 28 of 34 houses


,err_naive,err_model,gain_%
house_id,,,
17,0.086,0.099,-15.069
27,0.070,0.078,-10.615
9,0.264,0.276,-4.233
12,0.421,0.433,-2.665
5,0.166,0.169,-1.659
42,0.134,0.135,-1.028
29,0.471,0.470,0.219
40,0.549,0.547,0.459


In [7]:
imp = pd.Series(model.booster_.feature_importance(importance_type="gain"),
                index=FEATURES).sort_values()
px.bar(imp, orientation="h", title="Feature importance (gain)").update_layout(showlegend=False)

In [8]:
h = 17
w = test[test.house_id == h].loc["2019-05-13":"2019-05-19", ["kwh", "pred", "lag_24"]]
px.line(w, title=f"House {h}: actual vs forecast, one week in May 2019")